In [ ]:
import numpy as np
from scipy.io import wavfile
import matplotlib.pyplot as plt
# Load the recording
Fs, x = wavfile.read('/content/hallelujah_real_8k.wav')
x = x.astype(float) / 32768.0 # convert to floating point, range [-1, 1]

/tmp/ipykernel_6567/2574831005.py:5: WavFileWarning: Chunk (non-data) not understood, skipping it.
  Fs, x = wavfile.read('/content/hallelujah_real_8k.wav')


## Problem 1

In [ ]:
t_start = 0
window_s = 8
t_index = int(window_s * Fs)
t = np.linspace() # fill in the arguments
plt.figure(figsize=(8, 4))
plt.plot(t, x, '-')
plt.xlabel('Time (ms)')
plt.ylabel('x[n]')
plt.title('Time-domain signal')
plt.grid(True)
plt.savefig('problem1_time_domain.png', dpi=150)
plt.show()
print(f'Sampling period Ts = 1/Fs = {1/Fs*1000:.4f} ms')

## Problem 2

In [ ]:
# Extract the 3.5 ms window used in lecture
t_start = 0.850 # seconds
window_s = # put the window size here
n_start = int(t_start * Fs)
N = int(window_s * Fs)
xs = x[n_start : n_start + N]
n = np.arange(N)
t_ms = (n_start + n) / Fs * 1000
plt.figure(figsize=(8, 4))
plt.stem(t_ms, xs)
plt.xlabel('Time (ms)')
plt.ylabel('x[n]')
plt.title(f'Time-domain signal, zoomed to {window_s} ms')
plt.grid(True)
plt.savefig('problem2_sample_time_domain.png', dpi=150)
plt.show()
print(f'Number of samples N = {N}')
print(f'Sampling period Ts = 1/Fs = {1/Fs*1000:.4f} ms')

## Problem 3

In [ ]:
N = len(xs) # from Problem 1
X = np.fft.rfft(xs)

Y_0 = X[0]
Y_1 = np.abs(X)
A0 = Y_0.real / N
C = 2 * Y_1 / N
C[0] = A0
if N % 2 == 0:
  C[-1] = np.abs(X[-1]) / N
theta = np.angle(X)

k = np.arange(len(X)) # k = 0, 1, ..., N/2
F = k / N # normalized frequency, cycles/sample
f = F * Fs # actual frequency, Hz

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.stem(F, C, basefmt=' ')
ax1.set_xlabel('Normalized frequency F (cycles/sample)')
ax1.set_ylabel('$C_k$ (amplitude)')
ax1.set_xlim(0, 0.5)
ax1.grid(True)

ax2 = ax1.twiny()
ax2.set_xlim(0, Fs / 2)
ax2.set_xlabel('Frequency f = Fs . F (Hz)')

# second x-axis, same plot
plt.title('One-sided amplitude spectrum')
plt.savefig('problem3_spectrum.png', dpi=150)
plt.show()

## Problem 4

In [ ]:
K = 20 # <-- try a few different values

C_ac = C[1:] # amplitudes for k = 1, ..., N/2 (excludes A0)
theta_ac = theta[1:]
k_ac = k[1:]

keep = np.argsort(C_ac)[::-1][:K] # indices of the K largest amplitudes

# ---- Reconstruct using the cosine formula directly ----
x_recon = np.full(N, A0)
for i in keep:
  x_recon = x_recon + C_ac[i] * np.cos(2*np.pi*k_ac[i]*n/N + theta_ac[i])

# ---- Spectrum after keeping only K components, for plotting ----
C_reduced = np.zeros_like(C)
C_reduced[0] = A0
C_reduced[1 + keep] = C_ac[keep]

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.stem(F, C_reduced, basefmt=' ')
ax1.set_xlabel('Normalized frequency F (cycles/sample)')
ax1.set_ylabel(f'$C_k$ (K={K} components kept)')
ax1.set_xlim(0, 0.5)
ax1.grid(True)

ax2 = ax1.twiny()
ax2.set_xlim(0, Fs / 2)
ax2.set_xlabel('Frequency f = Fs . F (Hz)')

plt.title(f'One-sided spectrum after keeping K={K} coefficients')
plt.savefig('problem4_spectrum_reduced.png', dpi=150)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(t_ms, xs, label='Original')
plt.plot(t_ms, x_recon, '--', label=f'Reconstructed (K={K})')
plt.xlabel('Time (ms)')
plt.ylabel('x[n]')
plt.legend()
plt.grid(True)
plt.title('Original vs. reconstructed signal (cosine sum)')
plt.savefig('problem4_time_domain.png', dpi=150)
plt.show()